# The Rise of Digital Payments in Kenya


**Author:** Benard Musyoka Mwinzi  
**Date:** 07/03/2026

## Goal

*The goal of this project is to analyze how mobile money transformed Kenya’s payment ecosystem compared to traditional card transactions.*

The purpose of this section of the project is to prepare the dataset for analysis

## Project Part 1: Data Cleaning & Feature Engineering
By the end of this section we will have:
1. Loaded and inspected both raw datasets
2. Standardised column names, data types, and date formats
3. Engineered analytical features (growth rates, ratios, flags)
4. Merged both datasets into a single analysis-ready file
5. Exported three clean CSVs ready for EDA and database loading

### 1. Setup — Libraries & Configuration

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


### 2. Load Raw Data
We load both CSVs and perform an initial inspection — checking shape, dtypes, and a sample of rows.

In [2]:
mobile_raw = pd.read_csv("D:\\Python\\Digital Payments\\MobilePayments.csv")
cards_raw = pd.read_csv("D:\\Python\\Digital Payments\\NumberOfTransactions.csv")

print("📂 Mobile Payments Dataset")
print(f"   Shape: {mobile_raw.shape[0]} rows × {mobile_raw.shape[1]} columns")
print(f"   Columns: {list(mobile_raw.columns)}\n")


print("📂 Card Transactions Dataset")
print(f"   Shape: {cards_raw.shape[0]} rows × {cards_raw.shape[1]} columns")
print(f"   Columns: {list(cards_raw.columns)}")

📂 Mobile Payments Dataset
   Shape: 227 rows × 6 columns
   Columns: ['Year', 'Month', 'Active Agents', 'Total Registered Mobile Money Accounts (Millions)', 'Total Agent Cash in Cash Out (Volume Million)', 'Total Agent Cash in Cash Out (Value KSh billions)']

📂 Card Transactions Dataset
   Shape: 199 rows × 8 columns
   Columns: ['Year', 'Month', 'Prepaid Cards on ATMs', 'Charge Cards on ATMs', 'Credit Cards on ATMs', 'Debit Cards on ATMs', 'POS Machines (Debit,Credit,Prepaid, Charge Cards)', 'Total']


In [3]:
# Preview Mobile Payments
print("── Mobile Payments (first 5 rows) ──")
display(mobile_raw.head())

── Mobile Payments (first 5 rows) ──


,Year,Month,Active Agents,Total Registered Mobile Money Accounts (Millions),Total Agent Cash in Cash Out (Volume Million),Total Agent Cash in Cash Out (Value KSh billions)
0,2026,January,474786,90.41,215.08,699.64
1,2025,December,473536,89.46,217.58,722.53
2,2025,November,475079,89.06,213.68,665.14
3,2025,October,462760,87.94,218.44,678.98
4,2025,September,456742,87.01,210.41,684.82


In [4]:
# Preview Card Transactions
print("── Card Transactions (first 5 rows) ──")
display(cards_raw.head())

── Card Transactions (first 5 rows) ──


,Year,Month,Prepaid Cards on ATMs,Charge Cards on ATMs,Credit Cards on ATMs,Debit Cards on ATMs,"POS Machines (Debit,Credit,Prepaid, Charge Cards)",Total
0,2026,January,24499,0,44767,3548699.0,5071291,3617965
1,2025,December,24566,0,49025,3837343.0,5329305,3910934
2,2025,November,24476,0,47738,3530631.0,5126468,3602845
3,2025,October,34419,0,42287,3613810.0,5525897,3690516
4,2025,September,27721,0,47672,3541318.0,5586684,3616711


In [5]:
# Check data types
print("── Mobile Payments dtypes ──")
print(mobile_raw.dtypes)
print()
print("── Card Transactions dtypes ──")
print(cards_raw.dtypes)

── Mobile Payments dtypes ──
Year                                                   int64
Month                                                    str
Active Agents                                          int64
Total Registered Mobile Money Accounts (Millions)    float64
Total Agent Cash in Cash Out (Volume Million)        float64
Total Agent Cash in Cash Out (Value KSh billions)    float64
dtype: object

── Card Transactions dtypes ──
Year                                                   int64
Month                                                    str
Prepaid Cards on ATMs                                  int64
Charge Cards on ATMs                                   int64
Credit Cards on ATMs                                   int64
Debit Cards on ATMs                                  float64
POS Machines (Debit,Credit,Prepaid, Charge Cards)      int64
Total                                                  int64
dtype: object


### 3. Clean — Mobile Payments Dataset
Steps:
- Rename columns to snake_case for consistency
- Create a proper `date` column from Year + Month
- Convert numeric columns (handle any string formatting)
- Engineer derived features: agent efficiency, average transaction value, MoM growth

In [6]:
# Copying the raw data for cleaning
mobile = mobile_raw.copy()

In [7]:
# ── Renaming columns ────────────────────────────────────────────
mobile.columns = [
    "year", "month", "active_agents",
    "registered_accounts_millions",
    "cash_in_out_volume_millions",
    "cash_in_out_value_ksh_billions"
]

In [8]:
# ── Creating a date column for time series analysis ───────────────────────────────
mobile['date'] = pd.to_datetime(mobile['year'].astype(str) + '-' + mobile['month'].astype(str).str.zfill(2))

In [9]:
# ── Numeric conversion of variables ───────────────────────────────
num_cols = ["active_agents", "registered_accounts_millions", "cash_in_out_volume_millions", "cash_in_out_value_ksh_billions"]

for col in num_cols:
    mobile[col] = pd.to_numeric(
        mobile[col].astype(str).str.replace(",", ""), errors="coerce"
    )

In [10]:
# ── Feature engineering ──────────────────────────────────────

# How many registered accounts does each agent serve?
mobile["agents_per_million_accounts"] = (
    mobile["active_agents"] / mobile["registered_accounts_millions"]
).round(2)

# Average value per transaction (converts billions/millions → KSh)
mobile["avg_transaction_value_ksh"] = (
    (mobile["cash_in_out_value_ksh_billions"] * 1e9) /
    (mobile["cash_in_out_volume_millions"]    * 1e6)
).round(2)

# Month-on-month growth in transaction volume
mobile["mobile_volume_mom_pct"] = (
    mobile["cash_in_out_volume_millions"].pct_change() * 100
).round(2)

print("✅ Derived features created:")
print("   • agents_per_million_accounts  — agent network density")
print("   • avg_transaction_value_ksh    — mean transaction size (KSh)")
print("   • mobile_volume_mom_pct        — month-on-month volume growth (%)")

✅ Derived features created:
   • agents_per_million_accounts  — agent network density
   • avg_transaction_value_ksh    — mean transaction size (KSh)
   • mobile_volume_mom_pct        — month-on-month volume growth (%)


In [11]:
# ── Checking Missing values ─────────────────────────────────────
mobile.isnull().sum()

year                              0
month                             0
active_agents                     0
registered_accounts_millions      0
cash_in_out_volume_millions       0
cash_in_out_value_ksh_billions    0
date                              0
agents_per_million_accounts       0
avg_transaction_value_ksh         0
mobile_volume_mom_pct             1
dtype: int64

There is only one missing value on the mobile_volume_mom_pct variable.
It is understandable since the mobile_volume_mom_pct is obtained by geting the percentage change of mobile_volume_mom and the first row does not have a prior month.


### 4. Clean — Card Transactions Dataset
Same pipeline: rename, date creation, numeric conversion, feature engineering.

In [12]:
# Copying raw data
cards = cards_raw.copy()

In [13]:
# ── Renaming columns ────────────────────────────────────────────
cards.columns = ["year", "month", "prepaid_atm", "charge_atm", "credit_atm", "debit_atm","pos_total", "total_card_transactions"]

In [14]:
# ── Creating a date column for time series analysis ───────────────────────────────
cards['date'] = pd.to_datetime(cards['year'].astype(str) + '-' + cards['month'].astype(str).str.zfill(2))

In [15]:
# ── Numeric conversion ────────────────────────────────────────
num_cols_c = [
    "prepaid_atm","charge_atm","credit_atm","debit_atm",
    "pos_total","total_card_transactions"
]
for col in num_cols_c:
    cards[col] = pd.to_numeric(
        cards[col].astype(str).str.replace(",", ""), errors="coerce"
    )

In [16]:
# ── Feature engineering ──────────────────────────────────────

# Total ATM withdrawals (all card types combined)
cards["atm_total"] = (
    cards["prepaid_atm"] + cards["charge_atm"] +
    cards["credit_atm"]  + cards["debit_atm"]
)

# What % of card transactions happen at POS (vs ATM cash withdrawals)?
# Higher = more merchant payments, less cash dependency
cards["pos_share_pct"] = (
    cards["pos_total"] / (cards["total_card_transactions"]+ cards["pos_total"]) * 100
).round(2)

# Debit card dominance at ATMs
cards["debit_share_pct"] = (
    cards["debit_atm"] / cards["atm_total"] * 100
).round(2)

# Month-on-month growth in total card transactions
cards["card_txn_mom_pct"] = (
    cards["total_card_transactions"].pct_change() * 100
).round(2)

print("✅ Derived features created:")
print("   • atm_total         — total ATM withdrawals across all card types")
print("   • pos_share_pct     — POS share of all card transactions (%)")
print("   • debit_share_pct   — debit card share of ATM transactions (%)")
print("   • card_txn_mom_pct  — month-on-month growth in card transactions (%)")

✅ Derived features created:
   • atm_total         — total ATM withdrawals across all card types
   • pos_share_pct     — POS share of all card transactions (%)
   • debit_share_pct   — debit card share of ATM transactions (%)
   • card_txn_mom_pct  — month-on-month growth in card transactions (%)


In [17]:
# checking for missing values
cards.isnull().sum()

year                       0
month                      0
prepaid_atm                0
charge_atm                 0
credit_atm                 0
debit_atm                  0
pos_total                  0
total_card_transactions    0
date                       0
atm_total                  0
pos_share_pct              0
debit_share_pct            0
card_txn_mom_pct           1
dtype: int64

There is only one missing value on the card_txn_mom_pct variable.
It is understandable since the first month does not have a prior raw to use when calculating percentage change.


In [18]:
# 5. Merging the datasets
# Columns to carry forward from each dataset
card_keep = (
    ["date","year","month"] +
    num_cols_c +
    ["atm_total","pos_share_pct","debit_share_pct","card_txn_mom_pct"]
)
mobile_keep = (
    ["date"] + num_cols +
    ["agents_per_million_accounts","avg_transaction_value_ksh","mobile_volume_mom_pct"]
)

merged = pd.merge(cards[card_keep], mobile[mobile_keep], on="date", how="outer")
merged.sort_values("date", inplace=True)
merged.reset_index(drop=True, inplace=True)

# ── Flag columns ──────────────────────────────────────────────
merged["covid_period"] = merged["date"].between("2020-03-01", "2020-09-30")
merged["card_data_available"] = merged["total_card_transactions"].notna()

print(f"✅ Datasets merged successfully")
print(f"   Total rows         : {len(merged)}")
print(f"   Date range         : {merged['date'].min().date()} → {merged['date'].max().date()}")
print(f"   Mobile-only rows   : {(~merged['card_data_available']).sum()}  (Mar 2007 – Jun 2009)")
print(f"   Fully joined rows  : {merged['card_data_available'].sum()}  (Jul 2009 → present)")

✅ Datasets merged successfully
   Total rows         : 227
   Date range         : 2007-03-01 → 2026-01-01
   Mobile-only rows   : 28  (Mar 2007 – Jun 2009)
   Fully joined rows  : 199  (Jul 2009 → present)


## Summary Statistics
A quick statistical snapshot of the overlap period.

In [19]:
overlap = merged[merged["card_data_available"]].copy()

print("═" * 50)
print("SUMMARY — Card Transactions (Jul 2009 – Present)")
print("═" * 50)
print(f"  Peak monthly transactions  : {overlap['total_card_transactions'].max():>15,.0f}")
print(f"  Latest month transactions  : {overlap['total_card_transactions'].iloc[-1]:>15,.0f}")
print(f"  Average POS share          : {overlap['pos_share_pct'].mean():>14.1f}%")
print(f"  Latest POS share           : {overlap['pos_share_pct'].iloc[-1]:>14.1f}%")

print()
print("═" * 50)
print("SUMMARY — Mobile Payments (Mar 2007 – Present)")
print("═" * 50)
mo = merged  # use full mobile history
print(f"  Registered accounts (latest): {mo['registered_accounts_millions'].iloc[-1]:>12.2f}M")
print(f"  Active agents (latest)       : {mo['active_agents'].iloc[-1]:>12,.0f}")
print(f"  Monthly volume (latest)      : {mo['cash_in_out_volume_millions'].iloc[-1]:>12.2f}M txns")
print(f"  Avg transaction value (latest): KSh {mo['avg_transaction_value_ksh'].iloc[-1]:>9,.0f}")

══════════════════════════════════════════════════
SUMMARY — Card Transactions (Jul 2009 – Present)
══════════════════════════════════════════════════
  Peak monthly transactions  :      28,167,738
  Latest month transactions  :       3,617,965
  Average POS share          :           24.9%
  Latest POS share           :           58.4%

══════════════════════════════════════════════════
SUMMARY — Mobile Payments (Mar 2007 – Present)
══════════════════════════════════════════════════
  Registered accounts (latest):        90.41M
  Active agents (latest)       :      474,786
  Monthly volume (latest)      :       215.08M txns
  Avg transaction value (latest): KSh     3,253


## 7. Exporting the clean files
Three clean CSV files are saved to a folder called `Clean_Data` and are ready for:
- EDA & Visualisations  
- Forecasting  
- Database ingestion

In [20]:
mobile.to_csv("cleaned_mobile.csv", index=False)
cards.to_csv("cleaned_cards.csv",   index=False)
merged.to_csv("cleaned_merged.csv",  index=False)

print("✅ Clean files exported:")

✅ Clean files exported:
